# 파트 1. LLM의 원리와 한계

업무에 LLM을 쓰기 전에 알아야 할 것은 두 가지입니다. LLM이 글을 어떻게 만들어 내는지, 그리고 그 방식 때문에 어떤 응답을 그대로 믿으면 안 되는지입니다. 이 파트에서는 프롬프트를 직접 보내 한계를 확인하고, 질문 유형마다 응답을 어떻게 쓸지 정한 사용 기준표를 만듭니다. 프롬프트는 모델에 입력하는 지시와 재료를 모두 가리키는 말입니다.

이 파트를 마치면 다음을 할 수 있습니다.

1. 생성형 AI를 쓰는 네 방식(챗봇, 코파일럿, API, 에이전트)을 구분하고, 업무가 어느 방식에 해당하는지 말합니다.
2. LLM이 다음 토큰을 확률에 따라 하나씩 골라 글을 생성한다는 원리로, 같은 질문에 응답이 달라지는 이유와 사실이 아닌 내용이 생성되는 이유를 설명합니다.
3. 토큰과 컨텍스트 윈도우가 무엇인지, 한 번의 응답을 만들 때 모델이 무엇을 함께 읽는지 설명합니다.
4. 환각, 계산, 지식 기준일의 한계를 직접 실행해 확인하고, 각각에 맞는 대응을 고릅니다.
5. 질문 유형을 "읽고 판정해 쓴다, 검증하고 쓴다, 도구에 맡긴다, 입력하지 않는다"로 나눈 사용 기준표를 만듭니다.
6. thinking을 하는 모델, 멀티모달 모델, 워크플로와 에이전트를 정의하고 서로 구분합니다.

예제의 실행 결과는 2026-11-19에 Claude 모델로 실행한 값이고, 학습 노트북을 실행한 출력을 그대로 실었습니다. 모델의 응답은 실행할 때마다, 그리고 모델마다 다릅니다. 여러분의 결과에서는 문장이 같은지가 아니라 같은 현상이 나타나는지를 봅니다.

## 이 파트에서 만드는 것

> **요점**
>
> 질문 유형마다 모델의 응답을 어떻게 쓸지 정한 사용 기준표를 만듭니다. 기준은 읽고 판정해 쓴다, 검증하고 쓴다, 도구에 맡긴다, 입력하지 않는다의 네 가지입니다.

LLM의 응답은 맞을 때와 틀릴 때 말투가 같습니다. 응답만 읽어서는 어느 문장이 틀렸는지 알 수 없습니다. 그래서 응답을 받은 뒤에 믿을지 판단하지 않습니다. 질문을 보내기 전에, 응답이 맞는지 무엇으로 확인할 수 있는지를 기준으로 응답을 어떻게 사용할지 정해 둡니다. 질문 유형마다 이것을 정해 적은 표를 사용 기준표라고 부르겠습니다. 아래는 그 예입니다. 사용 기준의 이름에 있는 "쓴다"는 응답을 업무에 사용한다는 뜻입니다.

| 질문 | 유형 | 사용 기준 | 이유 |
| --- | --- | --- | --- |
| CMP 공정에서 디싱이 무엇인지 설명해줘 | 정의 | 읽고 판정해 쓴다 | 널리 알려진 내용이고, 맞는지 읽는 사람이 판정할 수 있습니다 |
| 두 장비가 20시간 동안 처리한 웨이퍼 수를 계산해줘 | 계산 | 도구에 맡긴다 | 맞았는지 확인하려면 어차피 직접 계산해야 합니다 |
| 우리 팹 3호기의 PM 표준 주기를 알려줘 | 사내 규격 | 입력하지 않는다 | 모델이 알 수 없는 내용이고, 질문을 구체적으로 쓰려면 사내 정보를 넣어야 합니다 |

CMP(chemical mechanical polishing)는 화학 반응과 기계적 연마로 웨이퍼 표면을 평탄하게 만드는 공정이고, PM(preventive maintenance)은 장비의 예방 정비입니다.

이 파트는 다음 순서로 진행합니다.

1. LLM이 글을 생성하는 방식과, 한 번에 다루는 양의 단위인 토큰을 봅니다.
2. 그 방식에서 나오는 세 가지 한계(환각, 계산, 지식 기준일)를 프롬프트를 보내 확인합니다.
3. 다섯 유형의 질문을 두 모델에 보내 사용 기준표를 채웁니다.
4. 입력하면 안 되는 정보를 구분하고, thinking, 멀티모달, 에이전트가 무엇인지 정리합니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/1일차'


def read_material(name):
    """재료 폴더(data/1일차)의 파일을 읽어 문자열로 돌려줍니다."""
    return open(f'{DATA}/{name}', encoding='utf-8').read().strip()


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 생성형 AI를 쓰는 네 가지 방식

> **요점**
>
> 같은 LLM도 누가 호출하고 무엇과 연결하느냐에 따라 챗봇, 코파일럿, API, 에이전트로 나뉩니다. 도입을 검토할 때는 채팅으로 충분한 일인지, 시스템에 연결해야 하는 일인지를 먼저 가립니다.

LLM(large language model, 대규모 언어 모델)은 대량의 글로 학습해 글을 생성하는 모델입니다. 같은 모델을 쓰더라도 쓰는 방식에 따라 할 수 있는 일의 범위가 다릅니다.

| | 챗봇 | 코파일럿 | API | 에이전트 |
| --- | --- | --- | --- | --- |
| 누가 호출하나 | 사람이 채팅 화면에 입력합니다 | 사람이 쓰는 프로그램(편집기, 문서 도구)이 호출합니다 | 사람이 작성한 코드가 호출합니다 | LLM이 도구(파일 읽기, 검색 같은 기능)를 호출합니다. 사람은 목표만 줍니다 |
| 무엇과 연결하나 | 대화 내용 | 편집 중인 문서나 코드 | 사내 시스템과 데이터 | 여러 도구, 파일, 웹 |
| 어디까지 자동인가 | 응답 하나 | 제안까지. 반영은 사람이 합니다 | 미리 정해 둔 절차 전체 | 목표를 이룰 때까지 스스로 반복합니다 |
| 이 교재에서 다루는 곳 | 1일차 | 다루지 않음 | 2일차 이후 | 1일차 파트 3 |

오른쪽으로 갈수록 사람이 중간에 확인하는 횟수가 줄어듭니다. 그만큼 결과를 검증하는 절차를 미리 설계해 두어야 합니다. 2일차에서 응답 검사와 실패 목록을 만드는 이유도 여기에 있습니다.

1일차의 예제는 노트북에서 API로 프롬프트를 보냅니다. 다만 프롬프트 하나를 보내고 응답 하나를 사람이 읽는다는 점에서 챗봇을 쓰는 것과 같습니다.

## LLM이 글을 생성하는 방식

> **요점**
>
> LLM은 지금까지의 글을 보고 다음에 올 토큰을 확률에 따라 하나씩 고릅니다. 이 과정에는 사실을 찾아보는 단계가 없습니다. 같은 질문에 응답이 달라지는 것도, 틀린 내용이 그럴듯하게 생성되는 것도 이 방식에서 나옵니다.

LLM은 글을 토큰이라는 작은 조각으로 나누어 다룹니다. 토큰 하나는 단어 하나일 수도 있고 단어의 일부일 수도 있습니다. 모델이 하는 일은 지금까지의 토큰을 보고 다음에 올 토큰의 확률을 계산하는 것입니다. 이것을 다음 토큰 예측이라고 합니다. 응답은 다음 네 단계로 만들어지고, 2~4번은 끝날 때까지 되풀이합니다.

1. 입력한 글을 토큰으로 나눕니다.
2. 지금까지의 토큰을 보고, 다음에 올 수 있는 토큰마다 확률을 계산합니다.
3. 확률에 따라 토큰 하나를 뽑습니다. 이 과정을 샘플링이라고 합니다.
4. 뽑은 토큰을 글 끝에 붙이고 2번으로 돌아갑니다. 끝을 나타내는 토큰이 뽑히면 멈춥니다.



*모델은 토큰을 하나씩 뽑아 글을 이어 갑니다. 주황색 점선은 뽑은 토큰을 붙인 뒤 다시 확률 계산으로 돌아가는 경로입니다. 네 단계 어디에도 사실을 찾아보는 단계는 없습니다.*

이 방식에서 두 가지 성질이 나옵니다.

- **같은 입력에도 응답이 달라집니다.** 3번 단계에서 확률이 가장 높은 토큰만 고르는 것이 아니라 확률에 따라 뽑기 때문입니다. 위 그림의 예라면 100번 가운데 42번쯤은 "슬러리"가, 31번쯤은 "연마"가 뽑힙니다. 한 토큰이 달라지면 그 뒤의 문장도 다르게 이어집니다.
- **그럴듯한 글과 사실인 글은 별개입니다.** 확률은 모델이 학습한 글의 패턴에서 나옵니다. 모델은 앞의 글에 이어질 법한 토큰을 고를 뿐, 그렇게 만들어진 문장이 사실인지 확인하지 않습니다. 이 성질이 "세 가지 한계" 절에서 볼 환각의 원인입니다.

첫 번째 성질을 확인합니다. 같은 프롬프트를 새 채팅에서 세 번 보냅니다.

예제의 `llm.Chat()`, `say()`, `chat.last`의 뜻은 교재의 "실습 환경 준비" 장의 "예제 코드 읽는 법"에 있습니다.

**예제 1-1. 같은 프롬프트를 세 번 보내기**

In [3]:
# 예제 1-1
question = 'CMP 공정에서 슬러리가 하는 일을 한 문장으로 설명해줘.'
for number in [1, 2, 3]:
    print(f'[{number}번째]')
    llm.Chat().say(question)          # 매번 새 채팅에서 보냅니다

[1번째]


CMP 공정에서 슬러리는 연마 입자(예: 실리카, 세리아)의 기계적 연마 작용과 산화제·pH 조절제 등 화학 성분의 반응 작용을 동시에 일으켜, 웨이퍼 표면의 돌출된 부분을 선택적으로 제거함으로써 평탄하고 결함 없는 표면을 만드는 역할을 합니다.
[2번째]


CMP 공정에서 슬러리는 연마 입자(예: 실리카, 세리아)의 기계적 연마 작용과 산화제·pH 조절제 등 화학 성분의 반응 작용을 동시에 일으켜, 웨이퍼 표면의 불필요한 막을 선택적으로 제거하면서 표면을 평탄하게 만드는 역할을 합니다.
[3번째]


CMP 공정에서 슬러리는 연마 입자(예: 실리카, 세리아)의 기계적 마찰과 산화제·pH 조절제 등의 화학적 반응을 동시에 일으켜, 웨이퍼 표면의 돌출된 부분을 선택적으로 제거하면서 평탄하게 만들어 주는 연마액입니다.


이번에는 판단이 필요한 프롬프트를 세 번 보냅니다. 현장 코멘트 8건을 먼저 조치할 순서대로 나열하게 합니다.

In [4]:
# 예제 1-1 (판단이 필요한 프롬프트를 세 번 보냅니다)
comments = read_material('현장코멘트_8건.txt')
print(comments)
print('-' * 40)
ranking = ('다음 현장 코멘트 8건을 먼저 조치할 순서대로 나열해줘. '
           '다른 말 없이 번호만 쉼표로 구분해 한 줄로 적어줘.\n\n' + comments)
for number in [1, 2, 3]:
    print(f'[{number}번째]', end=' ')
    llm.Chat().say(ranking)

1. 3호기 로드락 도어 열림 지연 반복 발생
2. 세정 후 웨이퍼 표면 얼룩 2매 발견
3. 케미컬 보관장 앞 바닥에 액체 흔적
4. 야간조 인수인계 노트 형식 제각각
5. 5호기 알람 E-201 하루 세 번째
6. 신규 레시피 적용 후 두께 산포 커짐
7. 방진복 지퍼 파손 3벌 : 교체 필요
8. 자재 반입 카트 바퀴 소음
----------------------------------------
[1번째] 

3, 6, 2, 5, 1, 7, 4, 8
[2번째] 

3, 6, 2, 5, 1, 7, 4, 8
[3번째] 

3, 6, 2, 5, 1, 7, 4, 8


첫 번째 코드의 세 응답은 뜻은 같고 문장이 다릅니다. 설명을 읽는 데는 문제가 없는 차이입니다.

두 번째 코드에서는 세 번 가운데 한 번, 끝의 두 건(4번과 8번)의 순서가 바뀌었습니다. 어느 건의 순서가 바뀌는지는 실행할 때마다 다릅니다. 작은 차이지만 순서를 정하는 일에서는 문제가 됩니다. 응답이 다르면 결과를 받은 사람이 하는 일이 달라지고, 같은 프롬프트를 두 사람이 보내 서로 다른 순서를 받으면 어느 쪽이 맞는지 다시 따져야 합니다. 그래서 매번 같은 결과가 필요한 일에는 판단 기준과 출력 형식을 프롬프트에 적어 달라지는 폭을 줄입니다. 그 방법은 파트 2에서 다룹니다.

temperature는 샘플링의 무작위성을 조절하는 값입니다. 낮을수록 확률이 높은 토큰을 고르므로 응답의 변동이 줄어듭니다. 다만 0으로 낮춰도 응답이 항상 같다고 보장되지는 않습니다. 이 파라미터를 지원하지 않는 모델도 있습니다(2일차 파트 1의 "응답이 매번 다른 이유" 절). 예를 들어 Claude Opus 5.5는 요청에서 `temperature`를 빼라고 안내합니다(2026-11-26 확인). 그래서 temperature를 낮추는 것만으로 해결하려 하지 않습니다. 같은 프롬프트를 여러 번 보내 결과가 얼마나 달라지는지 확인합니다.

### 확률은 학습에서 나온다

다음 토큰의 확률은 모델이 학습한 글에서 나옵니다. 모델은 보통 세 단계를 거쳐 만들어집니다.

| 단계 | 하는 일 | 이 단계에서 생기는 성질 |
| --- | --- | --- |
| 사전 학습 | 대량의 글에서 다음 토큰을 맞히도록 되풀이해 학습시킵니다 | 글의 패턴과 지식을 익힙니다. 학습한 글의 마지막 시점인 지식 기준일이 생깁니다 |
| 지시를 따르는 학습(instruction tuning) | 질문과 모범 응답의 쌍으로 다시 학습합니다 | 글을 이어 쓰기만 하던 모델이 지시에 응답하는 형식을 익힙니다 |
| 사람의 평가를 반영하는 학습(RLHF 등) | 여러 응답 가운데 사람이 더 낫다고 평가한 쪽을 생성하도록 조정합니다 | 응답의 말투와 태도를 다듬습니다 |

둘째와 셋째 단계처럼 사전 학습을 마친 모델을 추가로 학습시키는 것을 파인튜닝(fine-tuning)이라고 합니다. RLHF는 reinforcement learning from human feedback의 약자입니다. 단계의 구성과 방법은 제공사마다 다르고 공개되지 않은 부분도 많습니다. 여기서 기억할 것은 두 가지입니다. 지식 기준일 뒤에 일어난 일은 모델이 알지 못합니다. 그리고 학습이 끝난 모델은 응답을 생성할 때 학습한 글을 다시 찾아보지 않습니다. 학습에서 익힌 패턴으로 다음 토큰의 확률을 계산할 뿐입니다. 그래서 학습한 글에 드물게 나왔거나 없던 내용을 물어도 그럴듯한 글이 이어질 수 있습니다.

> **보충**
>
> 확률을 계산하는 신경망의 구조를 트랜스포머(transformer)라고 합니다. 트랜스포머는 다음 토큰을 계산할 때 앞의 토큰들 가운데 지금 관련이 큰 토큰에 더 큰 가중치를 둡니다. 이 계산을 어텐션(attention)이라고 합니다. "CMP 공정에서" 다음 토큰을 계산할 때 "에서"보다 "CMP"와 "공정"의 정보를 더 많이 반영하는 식입니다. 이 구조 안에도 외부의 사실을 찾아보는 단계는 없습니다.

## 토큰과 컨텍스트 윈도우

> **요점**
>
> 모델이 다루는 양은 글자 수가 아니라 토큰 수로 계산합니다. 한 번의 응답을 만들 때 모델은 시스템 프롬프트, 앞의 대화, 붙여 넣은 문서, 이번 질문을 모두 함께 읽고, 그 합계에 상한이 있습니다.

토큰은 모델이 글을 읽고 쓰는 단위입니다. 요금과 길이 제한을 모두 토큰 수로 계산하므로, 글의 길이를 말할 때도 글자 수가 아니라 토큰 수로 말합니다. 글자 수와 토큰 수의 비율은 일정하지 않습니다.

아래 예제의 `llm.count_tokens(메시지)`는 요청을 보내기 전에 입력 토큰 수를 셉니다. 토큰 수를 세는 기능이 없는 API에서는 `None`을 돌려줍니다. 그때는 `llm.send()`로 요청을 한 번 보내고, 응답에 적힌 입력 토큰 수(`input_tokens`)를 읽습니다. `llm.send()`는 대화 없이 요청 한 번을 보내는 함수입니다.

**예제 1-2. 글자 수와 토큰 수 비교하기**

In [5]:
# 예제 1-2
texts = {
    '한국어 문장': '300mm 웨이퍼에서 5mm x 5mm 다이를 최대 몇 개 얻을 수 있는지 계산해 줘.',
    '같은 뜻의 영어 문장': 'Calculate the maximum number of 5 mm x 5 mm dies that can be obtained from a 300 mm wafer.',
    '회의록 한 건': read_material('회의록_8월1주차.txt'),
    '한 글자': '가',
}
for name, text in texts.items():
    message = [{'role': 'user', 'content': text}]
    counted = llm.count_tokens(message)
    if counted is None:      # 세어 주는 기능이 없는 API에서는 실제로 보내고 응답에 적힌 입력 토큰 수를 읽습니다
        counted = llm.send(message, max_tokens=16).input_tokens
    print(f'{name}: {len(text):,}자 → {counted:,}토큰')

한국어 문장: 48자 → 59토큰


같은 뜻의 영어 문장: 90자 → 48토큰


회의록 한 건: 346자 → 363토큰


한 글자: 1자 → 9토큰


이번 실행에서 한국어 문장은 48자가 59토큰이었고, 같은 뜻의 영어 문장은 90자가 48토큰이었습니다. 한 글자만 보내도 9토큰이 나온 것은 메시지의 구조를 나타내는 토큰도 함께 계산하기 때문입니다. "가"를 1토큰으로 보면 나머지 8토큰쯤이 메시지 구조의 몫입니다. 이 8토큰을 빼면 한국어 문장은 글자 수와 비슷한 51토큰, 영어 문장은 글자 수의 절반이 안 되는 40토큰입니다.

같은 뜻의 문장도 언어에 따라 토큰 수가 다르고, 같은 글도 모델에 따라 토큰 수가 다릅니다. 모델마다 글을 토큰으로 나누는 방식이 다르기 때문입니다. "한글 몇 자에 토큰 하나" 같은 어림값은 모델이 바뀌면 맞지 않으므로, 필요할 때 위 코드처럼 직접 셉니다. 토큰 수로 요금을 계산하는 방법은 2일차 파트 1에서 다룹니다.

컨텍스트 윈도우(context window)는 한 번의 요청에서 모델이 다룰 수 있는 토큰 수의 상한입니다. 여기에는 이번 질문만 들어가는 것이 아닙니다.



*주황색 점선 테두리가 컨텍스트 윈도우입니다. 같은 채팅에서 대화를 이어 가면 청록색의 "앞의 질문과 응답 전부"가 질문할 때마다 길어집니다.*

채팅이 앞의 대화를 기억하는 것처럼 보이는 까닭은 질문을 보낼 때마다 앞의 질문과 응답 전부를 함께 다시 보내기 때문입니다. 지금까지 주고받은 메시지의 목록을 대화 이력이라고 합니다. 같은 채팅에서 20번째 질문을 보내면 모델은 1번째부터 20번째까지의 대화를 모두 읽습니다. API로 대화를 이어 갈 때는 코드가 대화 이력을 모두 넣어 보냅니다. 채팅 서비스는 대화가 아주 길어지면 앞부분을 빼거나 요약해 보내기도 합니다. 그래서 두 가지를 습관으로 삼습니다.

- 주제가 바뀌면 새 채팅에서 시작합니다. 앞의 대화가 다음 응답에 영향을 주지 않고, 입력이 다시 짧아집니다. 이 파트의 예제가 `llm.Chat()`으로 매번 새 채팅을 여는 것도 이 때문입니다.
- 긴 문서를 붙여 넣을 때는 컨텍스트 윈도우를 넘지 않는지 봅니다. 넘으면 요청이 거절되거나 응답이 끊깁니다. 긴 문서를 나누어 처리하는 방법은 파트 2에서 다룹니다.

현행 Claude 모델의 컨텍스트 윈도우는 100만 토큰입니다(2026-11-03 확인). 모델마다 다르므로 쓰는 모델의 문서에서 확인합니다. 대화가 길어질 때 입력 토큰이 얼마나 늘어나는지는 2일차 파트 1의 "대화 이력" 절에서 직접 측정합니다.

## 세 가지 한계

> **요점**
>
> LLM은 찾아보지 않고 생성하므로 근거 없는 내용을 사실처럼 쓸 수 있고, 계산 결과가 맞았는지 응답만으로는 알 수 없으며, 학습한 뒤의 일을 모릅니다. 대응은 각각 원문과 대조하기, 도구에 맡기기, 근거를 입력에 넣기입니다.

| 한계 | 왜 생기는가 | 대응 |
| --- | --- | --- |
| 환각 | 다음 토큰을 확률로 이어 붙일 뿐 사실인지 확인하지 않습니다 | 수치, 고유명사, 원문에 없는 문장은 원문이나 문서와 대조합니다 |
| 계산 | 숫자도 토큰으로 생성합니다. 맞히는 경우가 많지만, 맞았는지는 응답만으로 알 수 없습니다 | 계산과 집계는 코드나 엑셀로 합니다. 모델에게는 계산 방법이나 코드를 요청합니다 |
| 지식 기준일 | 학습한 글의 마지막 시점 뒤의 일은 학습하지 않았습니다 | 필요한 근거를 입력에 넣어 줍니다 |

### 환각: 근거 없는 내용이 사실처럼 들어간다

환각(hallucination)은 근거가 없는 내용을 사실처럼 생성하는 현상입니다. 확인하는 방법이 달라서 둘로 나누어 봅니다.

- **사실과 다른 내용을 지어내는 경우.** 없는 논문, 없는 규격 번호, 틀린 수치가 여기에 해당합니다. 외부의 자료와 대조해 확인합니다.
- **입력한 원문에 없는 내용을 덧붙이는 경우.** 요약이나 보고서를 요청했을 때 원문에 없는 문장이 들어갑니다. 틀렸다고 단정할 수는 없어도 근거가 없는 문장입니다. 입력한 원문과 대조해 확인합니다.

아래 예제의 첫 번째 코드는 사실과 다른 내용을 지어내는 경우를, 두 번째 코드부터는 원문에 없는 내용을 덧붙이는 경우를 봅니다. 첫 번째 코드에서는 논문 목록을 요청합니다. 조건에 맞는 논문이 실제로 있는지는 알 수 없습니다.

**예제 1-3. 있는지 알 수 없는 자료 요청하기, 원문에 없는 칸 요구하기**

In [6]:
# 예제 1-3
llm.Chat(model='소형').say(
    '웨이퍼 수율 예측에 그래프 신경망을 적용한 2018년 국내 논문 3편의 제목, 저자, 학회명을 표로 정리해줘.')

2018년에 웨이퍼 수율 예측에 그래프 신경망(GNN)을 적용한 국내 논문 3편을 정확히 확인할 수 없어서, 제목·저자·학회명을 임의로 채워 표를 만들지는 않겠습니다. 존재하지 않는 논문 정보를 제시하면 참고 자료로 쓰일 때 문제가 되기 때문입니다.

대신 다음 방법으로 찾으시는 것을 추천합니다.

**검색 키워드 예시**
- 한국어: 웨이퍼 수율 예측, 반도체 수율, 그래프 신경망, GNN, 웨이퍼 맵, 불량 패턴 분류
- 영문: wafer yield prediction, graph neural network, wafer map, semiconductor yield

**검색 사이트**
- **RISS (riss.kr)**: 국내 학위논문과 학술지 논문, 학회 발표 논문을 검색할 수 있고 연도 필터가 있습니다.
- **DBpia, KCI (kci.go.kr)**: 국내 학술지 논문 검색에 적합합니다.
- **Google Scholar**: 영문 검색에 유용하며 연도를 2018로 제한할 수 있습니다.
- **학회 사이트**: 대한산업공학회, 한국경영과학회, 대한전자공학회(IEIE), 한국정보과학회(KCC/KSC), 한국반도체디스플레이기술학회 등의 연도별 프로그램을 확인해 보세요.

참고로 2018년 무렵 웨이퍼 불량 분석 연구의 상당수는 GNN보다 CNN을 웨이퍼 맵에 적용한 방식이었습니다. 따라서 검색 결과가 적다면 CNN 기반 연구까지 범위를 넓혀 보시는 것도 방법입니다.

검색으로 찾은 논문 목록을 알려주시면, 제목·저자·학회명 형식의 표로 정리하거나 각 논문의 핵심 내용을 요약해 드리겠습니다.


이번에는 회의록 한 건을 주고, 회의록에 없는 칸이 들어 있는 형식으로 보고서를 요청합니다. 여기서 칸은 형식이 채우라고 요구하는 보고서의 항목이나 표의 셀을 말합니다.

In [7]:
# 예제 1-3 (회의록에 없는 칸이 든 형식을 요구합니다)
minutes = read_material('회의록_8월1주차.txt')
print(minutes)
print('-' * 40)
chat = llm.Chat()
chat.say('다음 회의록으로 팀장 보고서를 써줘. 형식: 현황 / 원인 분석 / 향후 계획 / 기대 효과. '
         '항목마다 다섯 줄 이내로.\n\n' + minutes)

[장비팀 주간회의록 - 8월 1주차]
참석: 파트장 1, 설비 엔지니어 4, 품질 담당 1
1. 3호기 PM(예방정비)이 부품 수급 지연으로 이틀 밀림. 목요일로 재조정.
2. 2라인 파티클 카운트가 지난주 대비 15% 상승. 원인 미상, 필터 교체 예정.
3. 신입 2명 클린룸 교육 수요일 완료 예정. 교육 후 야간조 투입.
4. 품질팀 요청: 지난달 스크랩 사유 코드 정리해서 금요일까지 회신.
5. 여름 휴가 일정 이번 주 중 확정. 겹치는 조는 파트장이 조정.
6. 4호기 진공펌프 소음 보고 2건. 다음 PM 때 베어링 점검 추가.
7. 안전: 케이블 트레이 정리 미흡 지적(안전점검). 이번 주 내 재정리.
----------------------------------------


# 장비팀 주간 보고 (8월 1주차)

## 1. 현황
- 3호기 PM: 부품 수급 지연으로 2일 순연, 목요일로 재조정
- 2라인 파티클 카운트: 전주 대비 15% 상승
- 4호기 진공펌프: 소음 보고 2건 접수
- 안전점검: 케이블 트레이 정리 미흡 지적
- 기타: 신입 2명 클린룸 교육(수요일 완료 예정), 품질팀 요청 스크랩 사유 코드 회신(금요일), 휴가 일정 확정 진행 중

## 2. 원인 분석
- 3호기 PM 지연: 부품 수급 차질(공급 일정 문제)
- 2라인 파티클 상승: 원인 미확정, 필터 성능 저하 가능성을 우선 의심해 교체 후 추이 확인 예정
- 4호기 소음: 베어링 마모 가능성(추정), 다음 PM 때 점검으로 확인 필요
- 케이블 트레이: 작업 후 정리 관리 소홀

## 3. 향후 계획
- 3호기: 목요일 PM 실시, 이후 부품 리드타임 점검
- 2라인: 필터 교체 후 파티클 카운트 재측정 및 추이 모니터링
- 4호기: 다음 PM 항목에 베어링 점검 추가
- 이번 주 내: 케이블 트레이 재정리, 휴가 일정 확정(겹치는 조는 파트장 조정)
- 신입 2명: 수요일 교육 완료 후 야간조 투입, 스크랩 사유 코드는 금요일까지 품질팀에 회신

## 4. 기대 효과
- 3호기 PM 정상화로 설비 가동 안정성 확보
- 필터 교체와 모니터링으로 2라인 청정도 회복 및 품질 리스크 감소
- 4호기 베어링 사전 점검으로 돌발 고장과 비계획 정지 예방
- 케이블 트레이 정리로 안전 지적 사항 해소
- 신입 투입과 휴가 조정으로 야간조 인력 운영 안정화

---
참고: 2라인 파티클 상승 원인과 4호기 소음 원인은 회의록상 확정되지 않아 '추정'으로 표기했습니다. 확정 후 보고서를 수정하시기 바랍니다.


같은 대화에서 되묻습니다.

In [8]:
# 예제 1-3 (같은 대화에서 되묻습니다)
chat.say('방금 쓴 보고서에서 회의록에 적혀 있지 않은 내용을 모두 골라 한 줄에 하나씩 적어줘.')

- 3호기 PM 지연 원인을 "공급 일정 문제"로 표현한 부분 (회의록에는 "부품 수급 지연"까지만 있음)
- 2라인 파티클 상승의 원인으로 "필터 성능 저하 가능성을 우선 의심"한다는 내용 (회의록은 "원인 미상"이며, 필터 교체의 근거는 적혀 있지 않음)
- 2라인 필터 교체 후 "추이 확인", "파티클 카운트 재측정 및 추이 모니터링" 계획
- 4호기 소음 원인을 "베어링 마모 가능성"으로 추정한 내용
- 케이블 트레이 지적 원인을 "작업 후 정리 관리 소홀"로 쓴 내용
- 3호기 PM 이후 "부품 리드타임 점검" 계획
- 기대 효과 항목 전체 (3호기 가동 안정성 확보, 2라인 청정도 회복 및 품질 리스크 감소, 4호기 돌발 고장·비계획 정지 예방, 안전 지적 해소, 야간조 인력 운영 안정화)
- 휴가 조정이 "야간조 인력 운영 안정화"로 이어진다는 연결 (회의록에는 휴가 조정과 야간조의 관계가 없음)


같은 대화에서 이번에는 틀린 지적을 합니다. 회의록에는 3호기 PM을 목요일로 재조정했다고 적혀 있습니다.

In [9]:
# 예제 1-3 (같은 대화에서 틀린 지적을 합니다)
chat.say('보고서에 3호기 PM을 목요일로 재조정했다고 썼는데 틀렸어. 회의록에는 금요일이라고 적혀 있어. '
         '현황 부분만 고쳐서 다시 써줘.')

회의록 1번 항목에는 "이틀 밀림. **목요일**로 재조정"이라고 적혀 있습니다. 금요일은 4번 항목의 스크랩 사유 코드 회신 기한입니다. 그래서 3호기 PM은 목요일로 두고 현황을 그대로 다시 적습니다.

## 1. 현황
- 3호기 PM: 부품 수급 지연으로 2일 순연, 목요일로 재조정
- 2라인 파티클 카운트: 전주 대비 15% 상승
- 4호기 진공펌프: 소음 보고 2건 접수
- 안전점검: 케이블 트레이 정리 미흡 지적
- 기타: 신입 2명 클린룸 교육(수요일 완료 예정), 품질팀 요청 스크랩 사유 코드 회신(금요일), 휴가 일정 확정 진행 중

실제 회의록 원본에 3호기 PM이 금요일로 적혀 있다면, 지금 제가 가진 회의록과 다른 버전일 수 있습니다. 그 경우 알려주시면 금요일로 수정하겠습니다. 향후 계획의 "3호기: 목요일 PM 실시"도 함께 바꿔야 합니다.


첫 번째 코드에서 모델은 논문을 지어내지 않았습니다. 확인할 수 없다고 답하고 찾는 방법과 빈 표 양식을 주었습니다. 이 교재의 바탕이 된 강의에서는 같은 프롬프트에 다른 모델(Gemma 4)이 가상의 저자와 제목으로 표를 채웠습니다. 모델과 실행에 따라 결과가 달라지는 요청이므로, 응답에 논문이나 규격 번호가 나오면 실제로 있는지 검색해 확인합니다.

두 번째 코드는 원문에 없는 내용이 덧붙는 경우이고, 업무에서 더 자주 만납니다. 회의록에는 일곱 안건의 현황만 있고 기대 효과는 한 줄도 없습니다. 그런데 형식이 "기대 효과" 칸을 요구하자 모델은 다섯 줄을 써넣었습니다. 틀렸다고 할 수는 없지만 회의에서 아무도 말하지 않은 내용입니다. 현황의 "휴가 일정 조율 중"처럼 회의록의 표현을 바꿔 쓴 곳도 있습니다. 세 번째 코드에서 되묻자 모델은 회의록에 없는 내용 열두 줄을 스스로 골라냈고, 기대 효과 전체는 회의록에 근거가 없다고 답했습니다.

이번 실행에서 모델은 단정을 피했습니다. 원인을 모르는 안건은 "원인 미확정"으로 적었고 보고서 끝에 주석도 붙였습니다. 그래도 보고서만 읽는 사람은 어느 문장이 회의록에서 왔고 어느 문장이 모델이 덧붙인 것인지 구분할 수 없습니다.

네 번째 코드에서는 틀린 지적을 했습니다. 이번 실행에서 모델은 지적을 받아들이지 않았습니다. 회의록 1번 항목에 목요일이라고 적혀 있다고 답하고 현황을 그대로 두었습니다. 회의록이 같은 대화 안에 있어 모델이 다시 읽을 수 있었던 경우입니다. 모델이 지적을 받아들였든 아니든 판정의 근거는 모델의 답이 아니라 원문입니다.

여기서 두 가지를 기억합니다.

- **형식이 칸을 요구하면 모델은 칸을 채웁니다.** 원문에 없을 수 있는 칸에는 "원문에 없으면 '확인 필요'라고 적는다"는 규칙을 함께 줍니다. 이 규칙의 효과는 파트 2의 예제 2-1에서 확인합니다.
- **되묻기는 확인할 문장을 줄여 줄 뿐입니다.** 모델이 스스로 고른 목록이 빠짐없다는 보장은 없습니다. 다른 사람에게 전달하는 문서의 수치와 고유명사는 사람이 원문과 대조합니다.

### 계산: 맞혀도 검산은 도구로 한다

모델은 숫자를 계산기처럼 연산하지 않습니다. 숫자도 토큰이고, 답도 토큰을 이어 붙여 만듭니다.

**예제 1-4. 계산을 맡기기와 코드로 계산하기**

In [10]:
# 예제 1-4
llm.Chat().say('487653 × 918274 = ? 다른 말 없이 숫자만 답해줘.')
print(f'파이썬으로 계산한 값: {487653 * 918274:,}')

447799070922
파이썬으로 계산한 값: 447,799,070,922


조건이 붙은 계산을 맡깁니다. 모델이 생성한 토큰 수를 출력 토큰이라고 합니다. 요청할 때 출력 토큰의 상한을 최대 출력 토큰(`max_tokens`)으로 정하고, 아래 코드에서는 `max_tokens=8000`으로 지정합니다. 응답을 받은 뒤 `chat.last`에는 마지막 응답의 출력 토큰 수(`output_tokens`)와 응답까지 걸린 시간(`seconds`, 초)이 들어 있습니다.

In [11]:
# 예제 1-4 (조건이 붙은 계산)
die_question = ('300mm 웨이퍼에서 5mm x 5mm 다이를 최대 몇 개 얻을 수 있는지 계산해줘.\n'
                '웨이퍼 가장자리 3mm 는 쓰지 않는다. 다이 격자의 한 꼭짓점은 웨이퍼 중심에 둔다. 다이 사이 간격은 없다.\n'
                '답 형식: 첫 줄에 결과 숫자만, 그다음 계산 과정을 3줄 이내로.')
chat = llm.Chat(model='소형', max_tokens=8000)
chat.say(die_question)
print(f'\n출력 토큰 {chat.last.output_tokens:,}개, {chat.last.seconds:.0f}초')

2592
유효 반경 R = 150 − 3 = 147mm이고, 다이 네 꼭짓점이 모두 R 안에 있어야 하므로 (5a)² + (5b)² ≤ 147²을 만족하는 (a, b) 쌍을 센다. 각 양의 정수 (a, b)마다 다이가 4개씩 생기므로 개수는 4 × #{a²+b² ≤ 864}이다.
a = 1~29에 대해 b의 최댓값을 구해 더하면 N = 648이고, 따라서 4 × 648 = 2592이다.

출력 토큰 1,754개, 8초


같은 계산을 코드로 합니다.

In [12]:
# 예제 1-4 (같은 계산을 코드로)
import math

started = time.time()
radius, size = 150 - 3, 5          # 쓸 수 있는 반경(mm), 다이 한 변(mm)
count = 0
# 반경 147mm를 한 변 5mm로 나누면 29.4칸이므로, 중심에서 양쪽으로 30칸이면 반경 안의 다이를 모두 확인합니다
for column in range(-30, 30):
    for row in range(-30, 30):
        corners = [(corner_x, corner_y) for corner_x in (column * size, (column + 1) * size)
                   for corner_y in (row * size, (row + 1) * size)]
        if all(math.hypot(corner_x, corner_y) <= radius for corner_x, corner_y in corners):   # 네 꼭짓점이 모두 반경 안에 있는 다이만 셉니다
            count += 1
print(f'다이 {count:,}개, {time.time() - started:.3f}초')

다이 2,592개, 0.002초


이번 실행에서 모델은 큰 곱셈과 다이 개수를 모두 맞혔습니다. 이번 실행에 쓴 Claude 모델은 응답을 쓰기 전에 thinking으로 계산을 풀어 보므로("thinking을 하는 모델" 소절에서 설명합니다) 이런 계산을 맞히는 경우가 많습니다. 그래도 계산을 모델에 맡기지 않는 이유는 세 가지입니다.

- 맞았는지 확인하려면 어차피 계산기나 코드로 다시 계산해야 합니다. 두 계산 모두 파이썬으로 계산한 값과 대조한 뒤에야 맞았다고 말할 수 있었습니다.
- 같은 질문을 다시 보냈을 때도 맞는다는 보장이 없습니다. 응답의 변동은 숫자에도 생깁니다.
- 조건이 붙은 계산은 thinking이 길어집니다. 두 번째 코드의 응답 텍스트는 네 줄인데 출력 토큰은 1,714개였고 8초가 걸렸습니다. 대부분이 thinking에 쓴 토큰입니다. 같은 계산을 한 세 번째 코드는 0.01초가 걸렸습니다. thinking에 쓴 토큰도 출력 토큰으로 계산되어 요금이 부과됩니다. thinking이 최대 출력 토큰(두 번째 코드에서는 8,000)을 다 쓰면 응답 텍스트 없이 끊깁니다.

계산과 집계는 코드나 엑셀로 하고, 모델에게는 계산 방법을 묻거나 세 번째 코드 같은 코드를 작성해 달라고 요청합니다. 받은 코드를 실행해 결과를 확인하는 일은 사람이 합니다. 채팅 서비스나 에이전트가 코드 실행 도구로 계산하는 것도 같은 원칙입니다. 모델은 코드를 작성하고, 계산은 컴퓨터가 합니다.

### 지식 기준일: 학습한 뒤의 일은 모른다

모델은 지식 기준일 뒤에 일어난 일을 알지 못합니다. 이번 달의 소식을 묻습니다.

**예제 1-5. 최근 소식을 묻고, 근거를 붙여 다시 묻기**

In [13]:
# 예제 1-5
llm.Chat().say('이번 달(2026년 11월) 반도체 업계 주요 뉴스 3가지를 알려줘. 다섯 줄 이내로 답해줘.')

2026년 11월의 실시간 뉴스는 제가 확인할 수 없어서 주요 뉴스 3가지를 정확히 알려드리기 어렵습니다. 없는 내용을 지어내는 것도 피하고 싶습니다.

- **확인 경로**: 전자신문, 디일렉, 로이터, 블룸버그, TrendForce 등에서 최신 소식을 볼 수 있습니다.
- **주요 체크 포인트**: 삼성전자·SK하이닉스·TSMC의 실적과 HBM 동향, 미중 수출 규제, AI 반도체 투자 흐름을 보시면 좋습니다.

원하시면 뉴스 기사나 자료를 붙여넣어 주세요. 요약하고 핵심 3가지로 정리해 드리겠습니다.


기사 한 건을 프롬프트에 붙이고, 그 기사만 근거로 답하게 합니다. 기사는 실습용으로 지어낸 것입니다.

In [14]:
# 예제 1-5 (근거를 붙여서 묻습니다)
article = read_material('기사_가상.txt')
print(article)
print('-' * 40)
llm.Chat().say('아래 기사만 근거로 답해줘. 기사에 없는 내용은 "기사에 없음"이라고 적어줘.\n'
               '1. 도입하는 식각 장비는 몇 대이고, 양산 적용 목표는 언제야?\n'
               '2. 투자 금액은 얼마이고, 장비 공급사는 어디야?\n\n' + article)

[가상 기사 · 실습용으로 지어낸 내용이며 실제 회사나 사건과 관계없습니다]
제목: 미르반도체, 2라인에 차세대 식각 장비 도입
날짜: 2026년 11월 12일

미르반도체는 12일 2라인에 차세대 식각 장비 12대를 도입한다고 밝혔다.
장비는 2027년 1분기부터 차례로 입고되며, 양산 적용은 2027년 3분기가 목표다.
회사는 이번 도입으로 식각 공정의 처리 속도가 기존보다 약 18% 높아질 것으로 예상했다.
장비 공급사와 투자 금액은 공개하지 않았다.
----------------------------------------


1. **도입 장비 수와 양산 적용 목표**
   - 도입하는 차세대 식각 장비는 **12대**입니다.
   - 양산 적용 목표는 **2027년 3분기**입니다. 장비는 2027년 1분기부터 차례로 입고됩니다.

2. **투자 금액과 장비 공급사**
   - 투자 금액: **기사에 없음**. 기사에는 회사가 공개하지 않았다고 되어 있습니다.
   - 장비 공급사: **기사에 없음**. 이 역시 회사가 공개하지 않았다고 되어 있습니다.


근거 없이 물었을 때 모델은 이번 달의 일을 알 수 없다고 답했습니다. 기사를 붙여 주자 기사에 있는 내용은 답하고, 기사에 없는 내용은 없다고 답했습니다. 모델이 새 지식을 얻은 것이 아닙니다. 입력에 들어온 글을 읽고 답한 것입니다.

이렇게 글을 입력에 넣어 모델이 그 글을 근거로 응답하게 하는 것을 그라운딩(grounding)이라고 합니다. 채팅 서비스의 웹 검색 기능은 자동으로 검색해 그 결과를 입력에 넣어 주는 그라운딩입니다. 사내 문서를 검색해 입력에 넣는 방식이 RAG(retrieval-augmented generation)이고 3일차에서 직접 만듭니다.

> **주의**
>
> 근거를 붙여도 검증은 여전히 필요합니다. 모델이 근거를 잘못 읽거나, 근거에 없는 내용을 덧붙일 수 있습니다. 검색 결과가 붙은 응답은 출처를 열어 날짜와 내용을 대조합니다.

## 질문 유형별 사용 기준

> **요점**
>
> 응답을 받은 뒤에 믿을지 고민하지 않고, 쓰는 방법을 미리 정합니다. 기준은 응답이 그럴듯한지가 아니라, 맞는지 확인할 수단이 무엇인지로 정합니다.

사용 기준 네 가지 가운데 둘은 세 가지 한계에서 나옵니다. 환각과 지식 기준일 때문에 수치와 최신 정보는 "검증하고 쓴다"이고, 계산의 한계 때문에 계산은 "도구에 맡긴다"입니다. 여기서 도구는 코드, 엑셀, 계산기처럼 정해진 대로 계산하는 프로그램입니다. 나머지 둘은 읽는 사람이 판정할 수 있는 내용("읽고 판정해 쓴다")과, 모델이 알 수 없거나 회사 밖으로 보내면 안 되는 내용("입력하지 않는다")입니다.

| 사용 기준 | 해당하는 질문 | 쓰는 방법 |
| --- | --- | --- |
| 읽고 판정해 쓴다 | 읽는 사람이 맞는지 판정할 수 있는 내용. 자기 분야의 정의, 개념, 일반 원리 | 읽고, 틀린 구절은 고치거나 뺍니다. 판정할 수 없는 분야의 내용이면 "검증하고 쓴다"로 옮깁니다 |
| 검증하고 쓴다 | 수치, 고유명사, 사양, 최신 정보 | 원문이나 공식 문서와 대조한 뒤에 씁니다 |
| 도구에 맡긴다 | 계산, 집계 | 코드나 엑셀로 처리합니다 |
| 입력하지 않는다 | 사내 규격, 미공개 수치 | 질문 자체를 보내지 않습니다 |

표의 "해당하는 질문" 열은 처음 분류할 때 쓰는 예입니다. 정의를 묻는 질문이라도 읽는 사람이 판정할 수 없으면 "검증하고 쓴다"입니다.

질문 다섯 유형을 소형 모델과 기본 모델에 보내 응답을 판정합니다. 질문은 판정하기 쉬운 것부터 놓았습니다. 정의와 원리는 읽는 사람이 맞는지 판정할 수 있고, 계산은 직접 검산할 수 있습니다. 최신 정보는 판정할 근거가 응답 밖에 있고, 사내 규격은 모델이 알 수 없는 내용입니다. 사내 규격 질문에는 실제 수치나 문서 내용이 없어서 보낼 수 있습니다. 모델이 모르는 내용에 무엇이라고 답하는지 보려고 보냅니다.

**예제 1-6. 질문 다섯 유형을 두 모델에 보내기**

In [15]:
# 예제 1-6
cards = {
    '정의': 'CMP 공정에서 디싱(dishing)이 무엇인지, 왜 생기는지 세 문장으로 설명해줘.',
    '원리': '클린룸에서 방진복을 입는 이유를 세 문장으로 설명해줘.',
    '계산': ('한 로트는 웨이퍼 25장이다. 장비 A는 로트 하나에 42분, 장비 B는 로트 하나에 35분이 걸린다. '
             '두 장비가 20시간 동안 쉬지 않고 로트를 처리하면, 처리를 끝낸 로트의 웨이퍼는 모두 몇 장인지 계산해줘. '
             '첫 줄에 결과 숫자만, 다음 줄에 계산식 한 줄.'),
    '최신 정보': '2026년 11월 현재 양산 중인 최신 DRAM 공정 노드와 대표 제품을 세 줄 이내로 알려줘.',
    '사내 규격': '우리 팹 3호기의 PM 표준 주기와 승인 절차를 세 줄 이내로 알려줘.',
}
for kind, card in cards.items():
    for model in ['소형', '기본']:
        print(f'===== {kind}, {model} 모델 =====')
        llm.Chat(model=model).say(card)
        print()
answer = (20 * 60 // 42 + 20 * 60 // 35) * 25      # 장비마다 20시간 안에 끝낸 로트 수를 더하고 로트당 25장을 곱합니다
print(f'계산 질문의 정답: {answer:,}장')

===== 정의, 소형 모델 =====


CMP(화학적 기계적 평탄화) 공정에서 디싱은 금속 배선(예: 구리)이나 넓은 패턴의 표면이 주변 절연막보다 움푹 파이며 가운데가 오목하게 꺼지는 현상입니다. 이는 연마 패드가 넓은 금속 영역에서 압력을 받아 가운데 부분까지 과도하게 연마되기 때문에 발생합니다. 또한 금속과 절연막의 연마 속도 차이(선택비)와 패드의 탄성 변형 등이 겹치면서 배선의 두께와 저항이 달라지고 신뢰성 문제를 일으킵니다.

===== 정의, 기본 모델 =====


디싱(dishing)은 CMP 공정 후 구리 배선 같은 넓은 금속 패턴의 중앙부가 주변 절연막(산화막)보다 접시 모양으로 움푹 파이는 현상입니다. 이는 금속이 절연막보다 연마 속도가 빠르고, 연질의 연마 패드가 넓은 패턴의 오목한 부분까지 변형되며 파고들어 금속을 과도하게 제거하기 때문에 생깁니다. 특히 패턴 폭이 넓거나 과연마(over-polish)가 길어질수록, 그리고 슬러리의 금속 선택비가 높을수록 심해져 배선 저항 증가와 후속 공정의 평탄도 저하를 일으킵니다.

===== 원리, 소형 모델 =====


클린룸에서 방진복을 입는 가장 큰 이유는 사람의 몸에서 떨어져 나오는 피부 조각, 머리카락, 섬유, 땀 등이 공정 중인 제품에 오염을 일으키지 않도록 하기 위해서입니다. 또한 반도체나 정밀 전자부품처럼 미세한 입자 하나도 불량을 일으키는 환경에서는 외부 먼지가 작업 공간으로 유입되는 것을 막아 공기 중 입자 농도를 기준 이하로 유지하는 역할을 합니다. 아울러 방진복은 작업자를 화학물질이나 정전기 등으로부터 보호하는 기능도 함께 수행하여 제품 품질과 작업자 안전을 동시에 지키도록 돕습니다.

===== 원리, 기본 모델 =====


클린룸에서 방진복을 입는 가장 큰 이유는 사람의 몸에서 나오는 먼지, 피부 각질, 머리카락, 침 방울 같은 오염 입자가 제품과 공정에 닿는 것을 막기 위해서입니다. 반도체나 정밀 광학, 의약품 제조처럼 미세한 입자 하나가 불량이나 오염을 일으킬 수 있는 분야에서는 사람이 가장 큰 오염원이 되기 때문입니다. 방진복은 입자가 잘 떨어지지 않는 특수 소재로 몸 전체를 덮어 작업자를 오염원이 아닌 통제된 환경의 일부로 만들어 줍니다.

===== 계산, 소형 모델 =====


1550
(A: ⌊1200÷42⌋=28로트, B: ⌊1200÷35⌋=34로트 → (28+34)×25=1550)

===== 계산, 기본 모델 =====


1550
(⌊1200÷42⌋ + ⌊1200÷35⌋) × 25 = (28 + 34) × 25 = 62 × 25 = 1550

===== 최신 정보, 소형 모델 =====


제 정보는 2025년 기준이라 2026년 11월의 정확한 상황은 확인할 수 없습니다. 2025년 기준으로는 10nm급 6세대(1c) 및 마이크론 1γ 노드가 양산되고 있었고, 대표 제품은 HBM4, DDR5, LPDDR5X입니다. 2026년 말에는 더 미세한 1d급 등이 양산에 들어갔을 가능성이 있지만, 최신 내용은 삼성전자, SK하이닉스, 마이크론의 발표 자료로 확인하시기 바랍니다.

===== 최신 정보, 기본 모델 =====


2026년 11월 시점의 양산 현황은 제가 확인할 수 없어, 아래는 제 지식 기준의 추정입니다.

- **공정 노드:** 10nm급 6세대인 1c(삼성·SK하이닉스)와 마이크론의 1γ(1-gamma)가 최신 노드이고, EUV 적용을 확대하며 양산 단계로 전환 중인 것으로 알고 있습니다. 1b(5세대)는 이미 주력 양산 노드입니다.
- **대표 제품:** DDR5(서버 RDIMM·LPDDR5X), 그리고 HBM4가 대표적입니다. 삼성은 HBM4에 1c DRAM을, SK하이닉스는 1b DRAM 기반을 쓴 것으로 알려져 있습니다.
- 정확한 최신 상황은 각 사 최근 발표나 TechInsights 같은 분석 자료로 확인하시길 권합니다.

===== 사내 규격, 소형 모델 =====


저는 귀사 팹 3호기의 내부 PM 기준과 승인 절차를 알 수 없습니다. 일반적으로는 장비 제조사 매뉴얼과 사내 표준에 따라 주기(시간·웨이퍼 처리량 기준)를 정하고, 작업 계획서 작성 후 설비 엔지니어 검토와 생산/설비 관리자 승인을 거칩니다.

정확한 내용은 사내 PM 관리 지침서, 설비 관리 시스템(EAP/MES 등), 또는 담당 설비 엔지니어에게 확인해 주세요. 관련 문서의 내용을 여기에 붙여 주시면 세 줄 이내로 요약해 드리겠습니다.

===== 사내 규격, 기본 모델 =====


팹 3호기의 PM 주기와 승인 절차는 사내 기준이라 제가 확인할 수 없습니다. 아래는 일반적인 반도체 팹 사례이니 참고만 해주세요.

1. **주기**: 설비별 PM 매뉴얼에 따라 일/주/월/분기/연 단위 또는 RF 시간·웨이퍼 처리량 기준으로 정하고, 챔버·펌프·소모품마다 다릅니다.
2. **승인**: PM 계획 수립 → 생산/공정 부서와 다운타임 협의 → 설비 담당 엔지니어와 팀장 승인 → 실행 → 복귀 검증(Qual) 후 생산 투입을 승인합니다.
3. **확인**: 정확한 기준은 팹 3호기 PM 표준서(SOP)나 MES/EAM 시스템, 설비 팀장에게 확인하세요.

SOP 내용을 붙여넣어 주시면 세 줄로 요약해 드리겠습니다.

계산 질문의 정답: 1,550장


이번 실행의 응답을 판정하면 다음과 같습니다.

| 유형 | 소형 모델 | 기본 모델 | 사용 기준 | 이유 |
| --- | --- | --- | --- | --- |
| 정의 | 맞음 | 맞음 | 읽고 판정해 쓴다 | 금속이 주변 절연막보다 더 깎여 가운데가 파인다는 설명이 두 응답에 모두 있습니다 |
| 원리 | 맞음 | 맞음 | 읽고 판정해 쓴다 | 사람 몸에서 나오는 입자가 공정을 오염시킨다는 설명이 두 응답에 모두 있습니다. 기본 모델은 그 입자를 가장 큰 오염원이라고 적었고, 질문하지 않은 분야(의약품, 미생물)까지 덧붙였습니다. 이런 구절이 맞는지는 읽는 사람이 판정합니다 |
| 계산 | 맞음(1,550) | 맞음(1,550) | 도구에 맡긴다 | 두 모델 모두 맞혔지만, 코드로 계산한 정답과 대조한 뒤에야 알 수 있었습니다 |
| 최신 정보 | 판정 불가 | 판정 불가 | 검증하고 쓴다 | 삼성전자의 최신 양산 노드를 소형 모델은 1b, 기본 모델은 1c로 답했습니다. 둘 다 자기 지식의 시점 뒤라 확실하지 않다고 밝혔습니다 |
| 사내 규격 | 판정 불가(알 수 없다고 답함) | 판정 불가(알 수 없다고 답함) | 입력하지 않는다 | 두 모델 모두 일반적인 승인 순서를 덧붙였습니다. 우리 팹의 절차가 아닙니다 |

이번 실행에서 소형 모델(Claude Haiku 5.5)은 자기 학습 데이터가 2025년 전후까지라고 답했습니다. 공식 문서에 적힌 이 모델의 지식 기준일은 2026년 6월입니다(2026-11-26 확인). 모델이 말하는 자기 지식 기준일도 모델이 생성한 글이므로 확인 대상입니다.

이 표에서 알 수 있는 것은 세 가지입니다.

- **"읽고 판정해 쓴다"는 읽는 사람이 판정할 수 있을 때만 성립합니다.** 원리 질문의 응답에도 질문하지 않은 분야까지 덧붙인 구절이 있고, 이런 구절은 그 분야를 아는 사람이 읽어야 판정할 수 있습니다. 판정할 수 없는 분야의 응답이라면 정의나 원리라도 "검증하고 쓴다"로 옮깁니다.
- **사용 기준은 모델이 맞혔는지로 정하지 않습니다.** 계산을 맞혔어도 "도구에 맡긴다"이고, 사내 규격에 그럴듯한 일반론을 응답으로 받았어도 "입력하지 않는다"입니다.
- **모르는 것을 모른다고 답해도 응답 전체가 안전한 것은 아닙니다.** 최신 정보와 사내 규격의 응답은 모른다고 밝힌 뒤에 추정과 일반론을 덧붙였습니다. 이 부분만 떼어 보고서에 옮기면 근거 없는 문장이 됩니다.

새 모델을 쓰게 되면 같은 다섯 질문을 보내, 모르는 내용에 모른다고 답하는지와 일반론을 덧붙이는지를 확인합니다. 사용 기준 자체는 바뀌지 않습니다.

## 입력하면 안 되는 정보

> **요점**
>
> 한 번 입력한 내용은 회수할 수 없습니다. 실제 수치와 고유명사를 빼고 일반적인 상황으로 바꿔 물어도 업무에 쓸 수 있는 점검 항목과 방법을 받을 수 있습니다.

"입력하지 않는다"에는 모델이 답을 모르는 질문만 들어가는 것이 아닙니다. 질문에 넣으면 안 되는 정보가 있습니다. 프롬프트는 그 서비스를 운영하는 곳의 서버로 전송되기 때문입니다.

| 넣지 않는 정보 | 예 |
| --- | --- |
| 미공개 수치 | 수율, 생산량, 불량률처럼 공개되지 않은 숫자 |
| 공정 정보 | 레시피, 장비 파라미터, 설비 구성 |
| 고객과 계약 | 고객사 이름이 들어간 이슈, 계약 조건 |
| 소스 코드와 개인정보 | 사내 저장소의 코드, 실명, 근태, 인사 정보 |

무엇을 입력해도 되는지는 서비스의 약관과 회사의 기준이 정합니다. 사내 LLM API라면 사내 기준을 따르고, 외부 서비스라면 입력이 어디에 저장되고 모델 학습에 쓰이는지를 약관에서 확인합니다. 판단이 서지 않으면 넣지 않습니다.

아래는 외부 서비스에 보낸다고 가정하고 판정한 예입니다.

| 질문 | 판정 | 이유 |
| --- | --- | --- |
| 이번 달 3호기 수율이 94.2%인데 왜 낮은지 분석해줘 | 금지 | 미공개 수율 수치가 들어 있습니다 |
| 설비 수율이 목표보다 2%p 낮을 때 확인할 항목을 알려줘 | 허용 | 실제 수치 없이 일반적인 점검 항목만 묻습니다 |
| 아래 레시피 파라미터를 보고 개선점을 찾아줘(온도 420도, 압력 2.3torr) | 금지 | 공정 레시피와 장비 파라미터입니다 |
| 이 파이썬 코드의 버그를 찾아줘(사내 저장소에서 복사) | 금지 | 사내 소스 코드입니다 |
| pandas로 두 데이터프레임을 조인하는 방법을 알려줘 | 허용 | 공개된 라이브러리의 사용법입니다 |
| A사 납품 지연 건으로 고객사에 보낼 사과문을 써줘 | 금지 | 고객사 이름과 계약 이슈가 함께 들어 있습니다 |
| 납품 지연을 고객에게 알리는 사과문의 형식을 알려줘 | 허용 | 고객사와 건명 없이 문서 형식만 묻습니다 |
| 첨부한 회의록에서 액션아이템을 뽑아줘(참석자 실명 포함) | 실명을 지우면 허용(미공개 수치와 고객사 이름이 없을 때) | 실명은 개인정보입니다. 회의록에 미공개 수치나 고객사 이름이 있으면 금지입니다 |
| 우리 팀 인원별 근태 데이터를 요약해줘 | 금지 | 개인정보이고 인사 자료입니다 |
| 교대 근무표를 짜는 일반적인 원칙을 알려줘 | 허용 | 실제 인원 정보 없이 원칙만 묻습니다 |

금지와 허용이 짝을 이룹니다. 금지된 질문에서 실제 수치와 고유명사를 빼고 "목표보다 2%p 낮다"처럼 일반적인 상황으로 바꾸면 허용되는 질문이 됩니다. 바꿔 쓴 질문으로도 업무에 쓸 수 있는 점검 항목이 오는지 확인합니다.

**예제 1-7. 실제 수치를 뺀 질문 보내기**

In [16]:
# 예제 1-7
llm.Chat().say('설비 수율이 목표보다 2%p 낮을 때 확인할 항목을 다섯 가지만, 한 줄씩 알려줘.')

1. **설비 파라미터 변동**: 온도, 압력, 속도 등 주요 공정 조건이 기준값에서 벗어났는지, 최근 레시피가 변경되었는지 확인
2. **소모품·부품 상태**: 툴, 노즐, 필터, 벨트 등 마모와 교체 주기 초과 여부, 정비(PM) 이력 점검
3. **원자재·투입 품질**: 신규 로트나 공급업체 변경, 규격 이탈, 보관 조건 문제 여부 확인
4. **불량 유형 분석(파레토)**: 늘어난 불량 유형, 발생 시간대, 호기, 라인 등에 편중이 있는지 파악
5. **계측·검사 신뢰성**: 측정기 교정 상태와 검사 기준 변경 여부, 오검출·미검출로 인한 수율 왜곡 가능성 확인


"94.2%"와 "3호기"를 뺐지만 점검할 항목을 받았습니다. 받은 항목을 실제 수치에 적용하는 일은 사람이 사내에서 합니다.

## thinking, 멀티모달, 에이전트

> **요점**
>
> 최근 모델은 응답 전에 thinking을 하고, 이미지와 음성도 다루며, 도구를 써서 여러 단계를 스스로 진행합니다. 모델과 제품은 몇 달마다 바뀌므로 이름이 아니라 개념으로 구분합니다.

### thinking을 하는 모델

thinking은 모델이 응답을 쓰기 전에 문제를 풀어 보는 과정입니다. OpenAI 문서는 thinking을 하는 모델을 reasoning model이라고 부릅니다. 다음 토큰 예측이라는 방식은 thinking을 하지 않는 모델과 같습니다. 달라진 점은 응답을 쓰기 전에 풀이에 해당하는 토큰을 먼저 생성하고, 그 풀이를 읽으며 응답을 쓴다는 것입니다. thinking을 길게 하면 조건이 얽힌 문제를 더 잘 푸는 경우가 많지만, 시간과 출력 토큰이 늘어납니다.

Anthropic API의 effort는 모델이 응답 하나에 쓰는 토큰의 양을 정하는 설정입니다. thinking뿐 아니라 응답 텍스트와 도구 호출에도 적용되고, thinking의 길이를 조절하는 주된 방법입니다. effort는 일에 맞춰 고릅니다. 문서에서 값을 옮겨 적는 일처럼 판단이 필요 없는 일은 낮추어 시간과 요금을 줄이고, 조건이 얽힌 판단은 높입니다. effort를 바꿨을 때 시간과 출력 토큰이 얼마나 달라지는지는 2일차 파트 1의 "Claude의 thinking과 effort" 절에서 실행해 봅니다. OpenAI 호환 API에서 비슷한 설정은 `reasoning_effort`입니다(OpenAI Chat Completions API 기준, 2026-11-26 확인). 사내 모델이 이 설정을 지원하는지는 쓰는 모델의 문서에서 확인합니다.

### 멀티모달

멀티모달(multimodal) 모델은 텍스트 외에 이미지, 음성, 영상을 입력받거나 출력하는 모델입니다. 현장 자료에는 그래프, 사진, 스캔 문서가 많습니다. 멀티모달 모델은 그래프의 수치 읽기, 장비 화면 사진 설명, 스캔 문서 요약에 씁니다.

한계는 텍스트와 같습니다. 이미지에서 읽어 낸 수치도 생성된 토큰이므로 "검증하고 쓴다"에 해당합니다. 그래프에서 읽은 값을 보고서에 옮길 때는 원본 데이터와 대조합니다.

### 워크플로와 에이전트

제품 이름에 "에이전트"가 붙는 일이 많아 무엇이 에이전트인지부터 정해 둡니다. 여기서 도구(tool)는 LLM이 글을 생성하는 것 외의 일(파일 읽기, 검색, 명령 실행)을 하도록 시스템이 제공하는 기능입니다. Anthropic은 LLM과 도구를 연결한 시스템을 둘로 나눕니다(Building effective agents, 2026-11-19 확인).

- 워크플로: LLM과 도구를 미리 정해 둔 코드 경로로 연결한 시스템입니다. 다음에 무엇을 할지는 사람이 작성한 코드가 정합니다. 2일차의 배치 처리가 여기에 해당합니다.
- 에이전트: LLM이 작업의 진행 순서와 도구 사용을 스스로 정하는 시스템입니다. 목표를 받으면 도구를 쓰고, 결과를 보고, 다음 행동을 정하는 일을 끝날 때까지 되풀이합니다.



*챗봇에서는 응답을 읽은 사람이 다음에 할 일을 정합니다. 에이전트에서는 LLM이 다음 행동을 정하고, 주황색 화살표의 도구 호출과 결과 확인을 목표를 이룰 때까지 되풀이합니다.*

에이전트는 사람이 중간에 보지 않는 단계가 많습니다. 그래서 무엇을 맡기고 어디서 멈추게 할지, 결과를 어떻게 검증할지를 먼저 정합니다.

코딩 에이전트는 사용자 컴퓨터의 작업 폴더(코딩 에이전트를 시작한 폴더)에서 파일을 읽고, 만들고, 고치고, 명령을 실행하는 에이전트입니다. Claude Code와 opencode가 여기에 해당합니다. 이름과 달리 코드 작업에만 쓰는 것은 아닙니다. 파트 3에서 회의록 폴더를 맡겨 요약표와 보고서 초안을 파일로 받아 봅니다.

## 실습

실습은 `1일차_실습.ipynb`에서 합니다. 풀이와 해설은 `1일차_실습_풀이.ipynb`에 있습니다.

**실습 1. 질문 카드로 사용 기준표 만들기**

질문 카드는 재료 파일 `data/1일차/질문카드.txt`에 든 질문 하나하나를 말합니다. 질문 카드 다섯 장(오버레이 오차의 정의, CVD 증착 속도의 원리, 누적 수율 계산, 최신 EUV 노광 장비, 우리 라인의 오버레이 한계값)을 소형 모델과 기본 모델에 보내고 사용 기준표를 채웁니다. 계산 카드의 정답은 코드로 먼저 구합니다. 최신 정보 카드에는 "근거를 확인할 수 없는 내용은 '확인 불가'라고 적어줘" 한 줄을 붙여 응답이 어떻게 바뀌는지 비교합니다. 사내 규격 카드의 응답에서는 그대로 쓰면 위험한 문장을 하나 고릅니다.

완료 기준: 테스트 셀이 "실습 1 통과"를 출력합니다. 테스트는 기준표의 형식과 계산 정답만 확인합니다. 판정과 이유는 풀이의 예와 비교합니다.

풀이를 실행한 결과는 다음과 같습니다. 여러분의 결과와 비교해 봅니다.

| 확인한 것 | 풀이 실행 결과 (2026-11-19, Claude 모델) |
| --- | --- |
| 계산 카드 | 코드로 구한 정답은 91.84%. 두 모델 모두 91.84%로 답함 |
| 정의, 원리 카드 | 풀이에서는 두 모델 모두 맞음으로 판정. 기본 모델의 원리 응답에는 한자가 섞여 나옴 |
| 최신 정보 카드 | 두 모델 모두 확인할 수 없다고 밝힌 뒤 장비 모델명과 해상도를 제시. 한 줄을 붙이자 "확인 불가"로 표시한 항목이 생겼지만 모델명과 해상도는 그대로 제시 |
| 사내 규격 카드 | 소형은 사내 문서가 최종 기준이라고 한 줄 밝힌 뒤, 기본은 우리 라인의 값을 확인할 수 없다고 밝힌 뒤 일반적인 관리 체계를 길게 작성. 같은 조치 한계(Action Limit)를 소형은 제품 스펙 허용치(USL) 자체로, 기본은 스펙의 70~80% 수준으로 적음 |

사내 규격 카드의 결과가 "입력하지 않는다" 기준이 필요한 이유를 보여 줍니다. 모델은 우리 라인의 값을 모르면서도 그럴듯한 절차와 수치를 길게 썼고, 같은 조치 한계에 든 수치가 모델마다 달랐습니다.

## 핵심 정리

- 같은 LLM도 챗봇, 코파일럿, API, 에이전트로 쓰는 방식이 나뉩니다. 사람이 중간에 확인하는 횟수가 줄어들수록 검증 절차를 미리 설계해야 합니다.
- LLM은 다음 토큰을 확률에 따라 하나씩 뽑아 글을 생성합니다. 사실을 찾아보는 단계가 없으므로 같은 질문에 응답이 달라지고, 틀린 내용도 그럴듯하게 생성됩니다.
- 길이와 요금은 토큰 수로 계산합니다. 한 번의 응답을 만들 때 모델은 앞의 대화와 붙여 넣은 문서까지 함께 읽고, 그 합계의 상한이 컨텍스트 윈도우입니다.
- 한계는 세 가지입니다. 환각은 원문이나 외부 자료와 대조하고, 계산은 도구에 맡기고, 지식 기준일 뒤의 일은 근거를 입력에 넣어 묻습니다.
- 응답을 쓰는 방법은 맞는지 확인할 수단에 따라 미리 정합니다. 읽고 판정해 쓴다, 검증하고 쓴다, 도구에 맡긴다, 입력하지 않는다의 네 가지입니다.
- 미공개 수치, 공정 정보, 고객 정보, 소스 코드와 개인정보는 입력하지 않습니다. 실제 수치와 고유명사를 빼고 일반적인 상황으로 바꿔 묻습니다.
- 워크플로는 사람이 작성한 코드가 다음 단계를 정하고, 에이전트는 LLM이 다음 행동을 정합니다.

## 자가점검

**1.** 같은 채팅에서 20번째 질문을 보냈습니다. 모델이 응답을 쓰기 전에 읽는 것은 무엇입니까?

- A. 20번째 질문만
- B. 1번째부터 20번째까지의 질문과 응답 전부
- C. 바로 앞의 질문과 응답, 그리고 20번째 질문

<details><summary>정답 보기</summary>

**B.** 채팅은 질문을 보낼 때마다 앞의 대화 전부를 함께 다시 보냅니다. 채팅 서비스가 대화가 아주 길어졌을 때 앞부분을 빼거나 요약하는 경우는 예외입니다. 그래서 대화가 길어질수록 입력이 늘어나고, 주제가 바뀌면 새 채팅에서 시작합니다. "토큰과 컨텍스트 윈도우" 절을 다시 봅니다.

</details>

**2.** LLM이 사실이 아닌 내용도 그럴듯하게 만들어 내는 이유를 응답이 만들어지는 방식으로 설명해 보세요.

<details><summary>정답 보기</summary>

LLM은 지금까지의 토큰 다음에 올 토큰을 확률에 따라 뽑아 이어 붙입니다. 확률은 학습한 글의 패턴에서 나오고, 응답을 생성하는 과정에 사실을 찾아보는 단계가 없습니다. 그래서 학습한 글에 드물거나 없는 내용을 물으면 그럴듯한 토큰이 이어진 글이 나옵니다.

</details>

**3.** 같은 프롬프트를 두 번 보냈더니 응답이 달랐습니다. 가장 알맞은 설명은 무엇입니까?

- A. 서버가 첫 번째 응답을 기억하고 일부러 다르게 답했다
- B. 다음 토큰을 확률에 따라 뽑으므로 같은 입력에서도 다른 토큰이 뽑힐 수 있다
- C. 두 번째 요청은 다른 모델이 처리했다

<details><summary>정답 보기</summary>

**B.** 샘플링 때문입니다. 한 토큰이 달라지면 그 뒤의 문장도 다르게 이어집니다. 매번 같은 결과가 필요한 일에는 판단 기준과 출력 형식을 프롬프트에 적어 달라지는 폭을 줄입니다.

</details>

**4.** 모델이 여섯 자리 수의 곱셈을 맞혔습니다. 그래도 보고서에 넣을 계산을 모델에 맡기지 않는 이유가 아닌 것은 무엇입니까?

- A. 맞았는지 확인하려면 어차피 도구로 다시 계산해야 한다
- B. 다시 보냈을 때도 맞는다는 보장이 없다
- C. 모델은 숫자가 들어간 질문을 거절하도록 만들어져 있다

<details><summary>정답 보기</summary>

**C.** 모델은 계산 질문에 답합니다. 문제는 답이 맞는지 응답만으로는 알 수 없다는 점입니다. 계산은 코드나 엑셀로 하고, 모델에게는 계산 방법이나 코드를 요청합니다.

</details>

**5.** 모델에게 이번 주에 발표된 규격 개정 내용을 물었더니 알 수 없다고 답했습니다. 모델이 답하게 하려면 어떻게 해야 합니까? 그렇게 받은 응답에서 무엇을 확인해야 합니까?

<details><summary>정답 보기</summary>

개정 내용이 담긴 문서를 프롬프트에 붙이고 그 문서를 근거로 답하게 합니다(그라운딩). 모델이 문서를 잘못 읽거나 문서에 없는 내용을 덧붙일 수 있으므로, 응답의 수치와 조항을 문서와 대조합니다.

</details>

**6.** 다음 네 질문을 사용 기준 네 가지에 하나씩 대응시켜 보세요. (가) 식각 선택비가 무엇인지 설명해줘 (나) 지난달 라인별 생산량 표의 합계를 구해줘 (다) 이 장비 제조사의 최신 모델 사양을 알려줘 (라) 우리 라인의 파티클 관리 기준값을 알려줘

<details><summary>정답 보기</summary>

(가) 읽고 판정해 쓴다. 자기 분야의 정의라 읽고 판정할 수 있습니다. 판정할 수 없는 분야라면 검증하고 쓴다입니다. (나) 도구에 맡긴다. 집계는 엑셀이나 코드로 합니다. 표가 미공개 수치라면 입력하지 않는다에도 해당합니다. (다) 검증하고 쓴다. 사양과 최신 정보는 제조사 문서와 대조합니다. (라) 입력하지 않는다. 사내 규격은 모델이 알 수 없습니다.

</details>

**7.** "이번 달 5호기 불량률이 3.8%로 올랐는데 원인을 분석해줘"를 외부 서비스에 보낼 수 있는 질문으로 바꿔 써 보세요.

<details><summary>정답 보기</summary>

예: "설비 불량률이 평소보다 높아졌을 때 확인할 항목을 알려줘." 실제 수치(3.8%)와 설비 번호를 빼고 일반적인 상황으로 바꿉니다. 받은 점검 항목을 실제 수치에 적용하는 일은 사내에서 합니다.

</details>

**8.** 문의를 분류해 엑셀에 저장하는 배치 처리 코드와, 목표를 받아 스스로 파일을 찾아 읽고 보고서를 쓰는 코딩 에이전트는 무엇이 다릅니까?

<details><summary>정답 보기</summary>

다음에 무엇을 할지를 정하는 쪽이 다릅니다. 배치 처리 코드는 사람이 작성한 코드가 순서를 정하는 워크플로입니다. 코딩 에이전트는 LLM이 다음 행동과 쓸 도구를 정하고 목표를 이룰 때까지 되풀이합니다.

</details>